# consistency check

Detailed workflow for this experiment. For catalogue inputs and dependencies, see [data](../../../docs/data.ipynb). Enable `LG_RUN_RESEARCH_WORKFLOW=1` before running the calculation cells.

In [ ]:
import os
RUN_RESEARCH_WORKFLOW = os.environ.get("LG_RUN_RESEARCH_WORKFLOW") == "1"
if not RUN_RESEARCH_WORKFLOW:
    raise RuntimeError("Optional workflow disabled. Read notebooks/92_reproduce_saved_analyses.ipynb and set LG_RUN_RESEARCH_WORKFLOW=1.")

In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file())
os.environ.setdefault("LG_PROJECT", str(ROOT / "research"))
PROJECT = Path(os.environ["LG_PROJECT"]).expanduser().resolve()
os.chdir(PROJECT)

def design_path(logical_name):
    if logical_name == "FINAL_EXPERIMENT_DESIGN_2026-10-01.md":
        return PROJECT / "out/final_stage_audit/portability_v3/original_design.txt"
    return PROJECT / logical_name

# Corrected TNG300 comparisons

Restart the kernel and run from the top. Set `LG_PROJECT` to your project directory on Roihu before running the setup cell. This notebook reads the corrected selection and existing feature caches; it does not require newly trained NPE checkpoints.

Inputs are verified against the v3 manifests. The corrected TNG sample has 1,530 pairs: 1,253 same-host and 277 different-host. Figures are written to `pickles_v3/figures/consistency_check`. Selection and feature caches are read only. Both simulations' radial pair velocities are **peculiar**; these comparison plots do not add a Hubble term.

Pair distributions count one row per pair. Galaxy-feature distributions explicitly distinguish unique final-pair members from pair-member entries (a galaxy may appear repeatedly). KS/energy/Wasserstein distances are descriptive; shared galaxies and hosts invalidate naive independent-row significance claims. Historical interpretation notes below require reassessment after rerunning.

Raw SOAP metadata, raw TNG catalogue checks, and large parent-catalogue diagnostics are optional flags in the setup cell and are off by default. Existing originals and outputs are preserved in the dated comparison-update backup.


In [ ]:
import os, sys
from pathlib import Path
PROJECT = Path(os.environ.get("LG_PROJECT", r"<research-project>"))
sys.path.insert(0, str(PROJECT / "analogues"))
from review_fixes import (RUN_VERSION, MODEL_DESCRIPTION, H0_BY_SIM, configure_run,
    feature_cache_dir, lg_observation, lg_to_peculiar, draw_noisy_feature,
    sample_lg_point, sample_lg_marginalized, calibration_summary, tng_volume_mpc3)
LEGACY_RESULTS_DIR = PROJECT / "pickles_v2"
FEATURE_CACHE_DIR = feature_cache_dir(PROJECT)


from comparison_utils import (load_comparison_inputs, finite_lookup,
    normalized_wasserstein, group_blocks, resample_blocks)
from splits import pair_groups, group_split
RESULTS_DIR, PLOT_DIR, tng_dir, TNG_DATA, TNG_MANIFEST = load_comparison_inputs(
    PROJECT, 'consistency_check')
RUN_SOAP_METADATA = False  # Optional remote metadata lookup; requires hdfstream.
RUN_TNG_CATALOGUE_CHECKS = False  # Optional raw catalogue; set TNG_BASE when enabled.
RUN_PARENT_CATALOGUE_CHECKS = False  # Optional large FLAMINGO parent-cache diagnostic.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import h5py
import sys
from pathlib import Path

# Comment out if you need equivalent TNG300 fields to FLAMINGO. They fetch particle level data from TNG300 api for all pair member subhalos and then compute
# a FLAMINGO equivalent quantity
# %run "<research-project>\analogues\compute_metallicity.py"
# %run "<research-project>\analogues\compute_kappa_corot.py"
# %run "<research-project>\analogues\compute_sigma_star.py"

tng_dir = PROJECT / "analogues/tng_for_comparison_v3"

tng_pairs_i = TNG_DATA["pairs_i"]
tng_pairs_j = TNG_DATA["pairs_j"]

tng_g_r_sample    = TNG_DATA["g_r"]
tng_met_sample    = TNG_DATA["metallicity"]
tng_vd_sample     = TNG_DATA["vel_disp"]
tng_mass_sample = TNG_DATA["stellar_mass"]
keep_idx = TNG_DATA["keep_idx"]

# pair-level
tng_gr_pairs  = np.concatenate([tng_g_r_sample[tng_pairs_i], tng_g_r_sample[tng_pairs_j]])
tng_met_pairs = np.concatenate([tng_met_sample[tng_pairs_i], tng_met_sample[tng_pairs_j]])
tng_vd_pairs  = np.concatenate([tng_vd_sample[tng_pairs_i],  tng_vd_sample[tng_pairs_j]])

print(f"TNG pair-member g-r count:  {np.isfinite(tng_gr_pairs).sum():,}")
print(f"TNG pair-level met count:  {np.isfinite(tng_met_pairs).sum():,}")
print(f"TNG pair-level vd count:   {np.isfinite(tng_vd_pairs).sum():,}")

In [ ]:
# Load cached final selections; remote SOAP metadata is optional.
from flamingo_lg_pipeline import FIELD_PATHS, open_default_soap
cache_dir = PROJECT / "analogues/cache/run_001_mstar2e10"
with open(cache_dir / "flamingo_sample.pkl", "rb") as f:
    sample = pickle.load(f)
with open(cache_dir / "flamingo_pairs.pkl", "rb") as f:
    pairs = pickle.load(f)
with open(FEATURE_CACHE_DIR / "tng_sigma_star.pkl", "rb") as f:
    tng_sigma_star = pickle.load(f)
pairs["have_same_host"] = pairs["have_same_host"].astype(bool)
flam_member_idx = np.unique(np.concatenate([pairs["i"], pairs["j"]]))
tng_member_idx = np.unique(np.concatenate([tng_pairs_i, tng_pairs_j]))
soap = open_default_soap() if RUN_SOAP_METADATA else None
print("Using cached final-pair members. SOAP metadata enabled:", RUN_SOAP_METADATA)

In [ ]:
flam_gr_pairs  = np.concatenate([sample["g_r"][pairs["i"]],         sample["g_r"][pairs["j"]]])
flam_met_pairs = np.concatenate([sample["metallicity"][pairs["i"]], sample["metallicity"][pairs["j"]]])
flam_vd_pairs  = np.concatenate([sample["vel_disp"][pairs["i"]],    sample["vel_disp"][pairs["j"]]])
flam_kap_pairs = np.concatenate([sample["kappa_rot"][pairs["i"]],   sample["kappa_rot"][pairs["j"]]])

# filter finite
flam_gr_pairs  = flam_gr_pairs[np.isfinite(flam_gr_pairs)]
flam_met_pairs = flam_met_pairs[np.isfinite(flam_met_pairs)]
flam_vd_pairs  = flam_vd_pairs[np.isfinite(flam_vd_pairs)]
flam_kap_pairs = flam_kap_pairs[np.isfinite(flam_kap_pairs)]

print(f"FLAMINGO pairs loaded: {len(pairs['i']):,}")
print(f"Keys: {list(pairs.keys())}")

# pair members
pairs_i = pairs["i"]
pairs_j = pairs["j"]

# g-r for each galaxy in each pair
flam_gr_i = sample["g_r"][pairs_i]
flam_gr_j = sample["g_r"][pairs_j]

print(f"FLAMINGO pair-member g-r count: {len(flam_gr_pairs):,}")
print(f"Median: {np.median(flam_gr_pairs):.3f}")
print(f"Blue fraction (g-r < 0.68): {np.mean(flam_gr_pairs < 0.68):.3f}")

> Historical interpretation from before the corrected comparison rerun. Reassess quantitative and causal claims using the new outputs.

### g-r color verification. We consider them approximately equal as both simulations use different methods but still the plots and statistics show good agreeability. FLAMINGO USES 9 GAMA bands: u,g,r,i,z,Y,J,H,K and TNG 300 uses 8 bands: U,B,V,K,g,r,i,z.


In [ ]:
# Unique galaxies participating in the current final pair selections.
flam_gr_valid = sample["g_r"][flam_member_idx]
flam_gr_valid = flam_gr_valid[np.isfinite(flam_gr_valid)]
print(f"\nFLAMINGO g-r field path: {FIELD_PATHS['gama_luminosity']}")
print(f"FLAMINGO valid g-r count: {len(flam_gr_valid):,}")
print(f"FLAMINGO g-r median: {np.median(flam_gr_valid):.3f}")
print(f"FLAMINGO g-r range:  {flam_gr_valid.min():.3f} – {flam_gr_valid.max():.3f}")
print(f"FLAMINGO blue fraction (g-r < 0.68): {np.mean(flam_gr_valid < 0.68):.3f}")

# TNG
tng_gr_valid = tng_g_r_sample[tng_member_idx]
tng_gr_valid = tng_gr_valid[np.isfinite(tng_gr_valid)]
print(f"\nTNG g-r:")
print(f"  Valid count: {len(tng_gr_valid):,}")
print(f"  Median: {np.median(tng_gr_valid):.3f}")
print(f"  Range:  {tng_gr_valid.min():.3f} – {tng_gr_valid.max():.3f}")
print(f"  Blue fraction (g-r < 0.68): {np.mean(tng_gr_valid < 0.68):.3f}")

# plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("g-r — unique final-pair members", fontsize=12)

bins = np.linspace(-0.5, 1.5, 40)

axes[0].hist(flam_gr_valid, bins=bins, density=True,
             alpha=0.6, color="#E07B3F",
             label=f"FLAMINGO (N={len(flam_gr_valid):,})")
axes[0].hist(tng_gr_valid, bins=bins, density=True,
             alpha=0.6, color="#4C72B0",
             label=f"TNG300 (N={len(tng_gr_valid):,})")
axes[0].axvline(0.68, color="gray", linestyle="--", label="threshold 0.68")
axes[0].set_xlabel("g-r")
axes[0].set_ylabel("Density")
axes[0].set_title("Distribution")
axes[0].legend(fontsize=9)

flam_gr_s = np.sort(flam_gr_valid)
tng_gr_s  = np.sort(tng_gr_valid)
axes[1].step(flam_gr_s, np.arange(1, len(flam_gr_s)+1)/len(flam_gr_s),
             color="#E07B3F", label="FLAMINGO", lw=1.5, where="post")
axes[1].step(tng_gr_s,  np.arange(1, len(tng_gr_s)+1)/len(tng_gr_s),
             color="#4C72B0", label="TNG300", lw=1.5, where="post")
axes[1].axvline(0.68, color="gray", linestyle="--", label="threshold 0.68")
axes[1].set_xlabel("g-r")
axes[1].set_ylabel("CDF")
axes[1].set_title("CDF")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_DIR / "verify_gr.pdf", dpi=150, bbox_inches="tight")
plt.show()

****With the correct sample sizes this time for g-r****


In [ ]:
from scipy.stats import ks_2samp, wasserstein_distance

# safety filter (idempotent if already clean)
flam_gr_pairs = flam_gr_pairs[np.isfinite(flam_gr_pairs)]
tng_gr_pairs  = tng_gr_pairs[np.isfinite(tng_gr_pairs)]

print(f"FLAMINGO pair-member g-r count: {len(flam_gr_pairs):,}")
print(f"TNG300   pair-member g-r count: {len(tng_gr_pairs):,}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("g-r Colour Verification — Pair-member entries", fontsize=12)

bins_gr = np.linspace(-0.5, 1.5, 40)
axes[0].hist(flam_gr_pairs, bins=bins_gr, density=True,
             alpha=0.6, color="#E07B3F",
             label=f"FLAMINGO (N={len(flam_gr_pairs):,})")
axes[0].hist(tng_gr_pairs, bins=bins_gr, density=True,
             alpha=0.6, color="#4C72B0",
             label=f"TNG300 (N={len(tng_gr_pairs):,})")
axes[0].axvline(0.68, color="gray", linestyle="--", label="threshold 0.68")
axes[0].set_xlabel("g-r")
axes[0].set_ylabel("Density")
axes[0].set_title("Distribution")
axes[0].legend(fontsize=8)

flam_gr_s = np.sort(flam_gr_pairs)
tng_gr_s  = np.sort(tng_gr_pairs)
axes[1].step(flam_gr_s, np.arange(1, len(flam_gr_s)+1)/len(flam_gr_s),
             color="#E07B3F", label="FLAMINGO", lw=1.5, where="post")
axes[1].step(tng_gr_s, np.arange(1, len(tng_gr_s)+1)/len(tng_gr_s),
             color="#4C72B0", label="TNG300", lw=1.5, where="post")
axes[1].axvline(0.68, color="gray", linestyle="--", label="threshold 0.68")
axes[1].set_xlabel("g-r")
axes[1].set_ylabel("CDF")
axes[1].set_title("CDF")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_08.pdf", bbox_inches="tight")
plt.show()

# KS + Wasserstein
ks, ks_p = ks_2samp(flam_gr_pairs, tng_gr_pairs)
wd       = wasserstein_distance(flam_gr_pairs, tng_gr_pairs)
pooled_std = np.sqrt(
    (len(flam_gr_pairs) * np.std(flam_gr_pairs)**2 +
     len(tng_gr_pairs)  * np.std(tng_gr_pairs)**2) /
    (len(flam_gr_pairs) + len(tng_gr_pairs))
)
print(f"\nKS stat:              {ks:.4f}  (descriptive distance; shared members)")
print(f"Wasserstein (raw):    {wd:.4f}")
print(f"Wasserstein (norm):   {wd/pooled_std:.4f}")

> Historical interpretation from before the corrected comparison rerun. Reassess quantitative and causal claims using the new outputs.

The histograms and CDFS overlap pretty nicely now we will confirm it using some statistics


In [ ]:
# checking negative values FLAMINGO more bluer in general with extreme values
n_negative = np.sum(flam_gr_valid < 0)
print(f"FLAMINGO negative g-r: {n_negative:,} ({n_negative/len(flam_gr_valid)*100:.2f}%)")

In [ ]:
from scipy.stats import gaussian_kde
import matplotlib.colors as mcolors

flam_gr_bal   = np.concatenate([sample["g_r"][pairs["i"]],           sample["g_r"][pairs["j"]]])
flam_mass_bal = np.concatenate([sample["stellar_mass"][pairs["i"]], sample["stellar_mass"][pairs["j"]]])
valid_f = np.isfinite(flam_gr_bal) & np.isfinite(flam_mass_bal) & (flam_mass_bal > 0)
flam_gr_bal   = flam_gr_bal[valid_f]
flam_logm_bal = np.log10(flam_mass_bal[valid_f])

tng_gr_bal   = np.concatenate([tng_g_r_sample[tng_pairs_i],  tng_g_r_sample[tng_pairs_j]])
tng_mass_bal = np.concatenate([tng_mass_sample[tng_pairs_i], tng_mass_sample[tng_pairs_j]])
valid_t = np.isfinite(tng_gr_bal) & np.isfinite(tng_mass_bal) & (tng_mass_bal > 0)
tng_gr_bal   = tng_gr_bal[valid_t]
tng_logm_bal = np.log10(tng_mass_bal[valid_t])

x_grid = np.linspace(10.2, 11.8, 100)
y_grid = np.linspace(0.3,  1.0,  100)
X, Y   = np.meshgrid(x_grid, y_grid)
rng    = np.random.default_rng(42)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Baldry-style Colour–Stellar Mass Diagram", fontsize=13)

datasets = [
    (axes[0], flam_logm_bal, flam_gr_bal, "#E07B3F", "FLAMINGO L1_m8", 50_000, 0.12),
    (axes[1], tng_logm_bal,  tng_gr_bal,  "#4C72B0", "TNG300-1",       len(tng_logm_bal), 0.20),
]

for ax, log_mass, g_r, color, label, n_kde, bw in datasets:
    hb = ax.hexbin(log_mass, g_r, gridsize=50,
                   cmap="Greys", mincnt=1,
                   norm=mcolors.LogNorm(),
                   linewidths=0.2)

    idx = rng.choice(len(log_mass), min(n_kde, len(log_mass)), replace=False)
    kde = gaussian_kde(
        np.vstack([log_mass[idx], g_r[idx]]),
        bw_method=bw
    )
    Z      = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)
    levels = np.percentile(Z[Z > 0], [50, 70, 88, 95])
    ax.contour(X, Y, Z, levels=levels,
               colors=color, linewidths=1.2, alpha=0.85)

    ax.axhline(0.68, color="gray", linestyle="--",
               linewidth=1.0, alpha=0.7, label="g-r = 0.68")
    ax.scatter(np.log10(5.5e10), 0.59, marker="*", s=200,
               color="gold", edgecolors="black", linewidths=0.5,
               zorder=5, label="MW (~0.59)")
    ax.scatter(np.log10(1.0e11), 0.63, marker="*", s=200,
               color="silver", edgecolors="black", linewidths=0.5,
               zorder=5, label="M31 (~0.63)")

    ax.set_xlabel(r"$\log_{10}(M_*/M_\odot)$", fontsize=11)
    ax.set_ylabel("g - r", fontsize=11)
    ax.set_title(f"{label}  (N={len(g_r):,})", fontsize=11)
    ax.set_xlim(10.2, 11.8)
    ax.set_ylim(0.3, 1.0)
    ax.legend(fontsize=9, loc="upper left")
    plt.colorbar(hb, ax=ax, label="Counts per hexbin (log scale)")

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_11.pdf", bbox_inches="tight")
plt.show()

> Historical interpretation from before the corrected comparison rerun. Reassess quantitative and causal claims using the new outputs.

Baldry style plot showing the two populations. Same threshold seems to work for both but plan is to experiment with different thresholds for NPE/Stan


In [ ]:
mu_f, mu_t = flam_gr_bal.mean(), tng_gr_bal.mean()
sd_f, sd_t = flam_gr_bal.std(), tng_gr_bal.std()

pooled_sd = np.sqrt((sd_f**2 + sd_t**2)/2)
effect = (mu_f - mu_t)/pooled_sd

print(f"Cohen's d = {effect:.3f}")

qs = [5, 25, 50, 75, 95]

print("FLAMINGO")
for q in qs:
    print(f"{q:2d}%: {np.percentile(flam_gr_bal, q):.3f}")

print("\nTNG300")
for q in qs:
    print(f"{q:2d}%: {np.percentile(tng_gr_bal, q):.3f}")


threshold = 0.68

f_blue = np.mean(flam_gr_bal < threshold)
t_blue = np.mean(tng_gr_bal < threshold)

print(f"FLAMINGO blue fraction: {f_blue:.3f}")
print(f"TNG300   blue fraction: {t_blue:.3f}")
print(f"Difference: {f_blue - t_blue:.3f}")

FLAMINGO has a greater bluer population. But the difference still seems to be pretty close.


### Metallicity Verification


In [ ]:
# Optional raw-catalogue definition check on current unique pair members.
if RUN_TNG_CATALOGUE_CHECKS:
    import illustris_python as il
    base_path = os.environ.get("TNG_BASE")
    if not base_path:
        raise ValueError("Set TNG_BASE to the directory containing groups_099 before enabling this check.")
    z_half_all = il.groupcat.loadSubhalos(base_path, 99, fields="SubhaloStarMetallicityHalfRad")
    z_custom = tng_met_sample[tng_member_idx]
    z_half = z_half_all[keep_idx[tng_member_idx]]
    valid = np.isfinite(z_custom) & np.isfinite(z_half) & (z_custom > 0) & (z_half > 0)
    ratio = z_half[valid] / z_custom[valid]
    corr = np.corrcoef(z_custom[valid], z_half[valid])[0, 1]
    print("TNG half-radius catalogue metallicity / custom all-bound-star metallicity")
    print(f"Unique current members used: {valid.sum():,}; median ratio: {np.median(ratio):.4f}; correlation: {corr:.4f}")
else:
    print("Optional raw TNG catalogue comparison skipped; cached CDFs remain available.")

In [ ]:
if RUN_SOAP_METADATA:
    print(f"\nFLAMINGO metallicity field path: {FIELD_PATHS['metallicity']}")
    
    met_field = soap[FIELD_PATHS["metallicity"]]
    print(f"\nFLAMINGO metallicity metadata:")
    for attr in met_field.attrs:
        print(f"  {attr}: {met_field.attrs[attr]}")
    
else:
    print('SOAP metallicity metadata skipped (optional).')

# pair-level
flam_met_pairs = np.concatenate([
    sample["metallicity"][pairs["i"]],
    sample["metallicity"][pairs["j"]]
])
flam_met_pairs = flam_met_pairs[np.isfinite(flam_met_pairs) & (flam_met_pairs > 0)]

tng_met_sample = TNG_DATA["metallicity"]
tng_met_pairs  = np.concatenate([
    tng_met_sample[tng_pairs_i],
    tng_met_sample[tng_pairs_j]
])
tng_met_pairs = tng_met_pairs[np.isfinite(tng_met_pairs) & (tng_met_pairs > 0)]

Z_sun_tng  = 0.0127   # Wiersma et al.  
print(f"\nFLAMINGO metallicity (raw):")
print(f"  N:      {len(flam_met_pairs):,}")
print(f"  Median: {np.median(flam_met_pairs):.6f}")
print(f"  Range:  {flam_met_pairs.min():.6f} – {flam_met_pairs.max():.6f}")

print(f"\nTNG metallicity (raw):")
print(f"  N:      {len(tng_met_pairs):,}")
print(f"  Median: {np.median(tng_met_pairs):.6f}")
print(f"  Range:  {tng_met_pairs.min():.6f} – {tng_met_pairs.max():.6f}")

print(f"\nTNG median in solar units: {np.median(tng_met_pairs)/Z_sun_tng:.3f} Z_sun")
print(f"Check FLAMINGO metadata for Z_sun used")

# plot
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Metallicity Verification — Pair-member entries", fontsize=12)

# raw distributions
bins_met = np.linspace(0, 0.06, 40)
axes[0].hist(flam_met_pairs, bins=bins_met, density=True,
             alpha=0.6, color="#E07B3F",
             label=f"FLAMINGO (N={len(flam_met_pairs):,})")
axes[0].hist(tng_met_pairs,  bins=bins_met, density=True,
             alpha=0.6, color="#4C72B0",
             label=f"TNG300 (N={len(tng_met_pairs):,})")
axes[0].set_xlabel("Metallicity Z (raw)")
axes[0].set_ylabel("Density")
axes[0].set_title("Raw comparison")
axes[0].legend(fontsize=8)

# solar units
axes[1].hist(flam_met_pairs / Z_sun_tng, bins=np.linspace(0, 5, 40),
             density=True, alpha=0.6, color="#E07B3F",
             label=f"FLAMINGO")
axes[1].hist(tng_met_pairs  / Z_sun_tng, bins=np.linspace(0, 5, 40),
             density=True, alpha=0.6, color="#4C72B0",
             label=f"TNG300")
axes[1].set_xlabel(r"Metallicity [$Z_\odot$]")
axes[1].set_ylabel("Density")
axes[1].set_title("Solar units (Z_sun = 0.0127)")
axes[1].legend(fontsize=8)

# CDFs
flam_met_s = np.sort(flam_met_pairs)
tng_met_s  = np.sort(tng_met_pairs)
axes[2].step(flam_met_s, np.arange(1, len(flam_met_s)+1)/len(flam_met_s),
             color="#E07B3F", label="FLAMINGO", lw=1.5, where="post")
axes[2].step(tng_met_s,  np.arange(1, len(tng_met_s)+1)/len(tng_met_s),
             color="#4C72B0", label="TNG300", lw=1.5, where="post")
axes[2].set_xlabel("Metallicity Z (raw)")
axes[2].set_ylabel("CDF")
axes[2].set_title("CDF")
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_17.pdf", bbox_inches="tight")
plt.show()

# KS + Wasserstein
from scipy.stats import ks_2samp, wasserstein_distance
ks, _ = ks_2samp(flam_met_pairs, tng_met_pairs)
wd    = wasserstein_distance(flam_met_pairs, tng_met_pairs)
pooled_std = np.sqrt(
    (len(flam_met_pairs) * np.std(flam_met_pairs)**2 +
     len(tng_met_pairs)  * np.std(tng_met_pairs)**2) /
    (len(flam_met_pairs) + len(tng_met_pairs))
)
print(f"\nKS stat:              {ks:.4f}")
print(f"Wasserstein (raw):    {wd:.6f}")
print(f"Wasserstein (norm):   {wd/pooled_std:.4f}")

The histograms seem pretty different along with the CDF. The KS stat and Wasserstein (norm) shows significant difference between the distributions despite seeming to have similar definitions. Below we test the correlation with logZ and try to explain the difference between the distributions


In [ ]:
# log-log correlation
flam_z_i = sample["metallicity"][pairs["i"]]
flam_mass_for_met = sample["stellar_mass"][pairs["i"]]
valid = np.isfinite(flam_mass_for_met) & np.isfinite(flam_z_i) & (flam_z_i > 0)
corr_loglog = np.corrcoef(np.log10(flam_mass_for_met[valid]), np.log10(flam_z_i[valid]))[0, 1]
print(f"FLAMINGO logM*-logZ correlation (pair i-side): {corr_loglog:.3f}")

# Reference relation among unique galaxies in the final FLAMINGO pair selection.
member_mass = sample["stellar_mass"][flam_member_idx]
member_z = sample["metallicity"][flam_member_idx]
full_valid = np.isfinite(member_mass) & (member_mass > 0) & np.isfinite(member_z) & (member_z > 0)
full_logm = np.log10(member_mass[full_valid])
full_logz = np.log10(member_z[full_valid])
corr_full = np.corrcoef(full_logm, full_logz)[0, 1]
print(f"FLAMINGO unique final-pair member logM*-logZ correlation (N={full_valid.sum():,}): {corr_full:.3f}")

tng_mass_pairs_idx = np.concatenate([tng_pairs_i, tng_pairs_j])
tng_logm_pairs = np.log10(tng_mass_sample[tng_mass_pairs_idx])
tng_logm_pairs = tng_logm_pairs[np.isfinite(tng_logm_pairs)]
print(f"TNG pair sample median logM*: {np.median(tng_logm_pairs):.3f}")

In [ ]:
# centered fit to remove conditioning issue
x_centered = full_logm - full_logm.mean()
slope_c, intercept_c = np.polyfit(x_centered, full_logz, 1)
print(f"Centered-fit slope: {slope_c:.3f}")

# non-parametric cross-check: binned median Z vs M*
bins = np.linspace(full_logm.min(), full_logm.max(), 15)
bin_idx = np.digitize(full_logm, bins)
bin_centers, bin_medians = [], []
for b in range(1, len(bins)):
    mask = bin_idx == b
    if mask.sum() > 100:
        bin_centers.append(full_logm[mask].mean())
        bin_medians.append(np.median(full_logz[mask]))
        
for c, m in zip(bin_centers, bin_medians):
    print(f"  logM*={c:.2f}: median logZ={m:.3f}  (N bin)")

In [ ]:
flam_median_logm = np.median(flam_logm_bal)
tng_median_logm  = np.median(np.log10(tng_mass_sample[np.concatenate([tng_pairs_i, tng_pairs_j])]))
if len(bin_centers) < 2:
    raise ValueError("Too few populated metallicity bins for the reference relation.")
within_bins = min(bin_centers) <= min(flam_median_logm, tng_median_logm) and max(flam_median_logm, tng_median_logm) <= max(bin_centers)
predicted_delta_logz = (np.interp(flam_median_logm, bin_centers, bin_medians) - np.interp(tng_median_logm, bin_centers, bin_medians)) if within_bins else np.nan
if not within_bins:
    print("One median mass lies outside the fitted bins; no extrapolated offset reported.")
observed_delta_logz = np.log10(np.median(flam_met_pairs) / np.median(tng_met_pairs))

print(f"Binned-relation estimate of Z offset: {predicted_delta_logz:.3f} dex")
print(f"Observed Z offset (FLAMINGO vs TNG):                    {observed_delta_logz:.3f} dex")
offset_ratio = predicted_delta_logz / observed_delta_logz if not np.isclose(observed_delta_logz, 0) else np.nan
print(f"Binned-relation estimate / observed offset: {offset_ratio:.3f} (descriptive, not a causal fraction)")

In [ ]:
# Metallicity: log10(Z/Z_sun) + per-simulation z-score

# log-transform (same Z_sun for both, since both store dimensionless Z)
flam_met_dex = np.where(sample["metallicity"] > 0,
                         np.log10(sample["metallicity"] / Z_sun_tng), np.nan)
tng_met_dex  = np.where(tng_met_sample > 0,
                         np.log10(tng_met_sample / Z_sun_tng), np.nan)

# fit separate scalers, each on their own pair-level distribution
_flam_pairs = np.concatenate([flam_met_dex[pairs["i"]], flam_met_dex[pairs["j"]]])
_tng_pairs  = np.concatenate([tng_met_dex[tng_pairs_i], tng_met_dex[tng_pairs_j]])

_ff = np.isfinite(_flam_pairs)
_tf = np.isfinite(_tng_pairs)

mu_flam,  std_flam  = np.mean(_flam_pairs[_ff]), np.std(_flam_pairs[_ff],  ddof=1)
mu_tng,   std_tng   = np.mean(_tng_pairs[_tf]),  np.std(_tng_pairs[_tf],   ddof=1)

# z-score each sim with its own scaler
flam_met_scaled = (flam_met_dex - mu_flam) / std_flam
tng_met_scaled  = (tng_met_dex  - mu_tng)  / std_tng

# sanity — both should be ~0, ~1
_fs = np.concatenate([flam_met_scaled[pairs["i"]], flam_met_scaled[pairs["j"]]])
_ts = np.concatenate([tng_met_scaled[tng_pairs_i], tng_met_scaled[tng_pairs_j]])
print(f"FLAMINGO scaled — mean: {np.nanmean(_fs):.4f}  std: {np.nanstd(_fs):.4f}")
print(f"TNG     scaled — mean: {np.nanmean(_ts):.4f}  std: {np.nanstd(_ts):.4f}")
print(f"\nFLAMINGO scaler: mu={mu_flam:.4f} dex  std={std_flam:.4f} dex")
print(f"TNG     scaler: mu={mu_tng:.4f}  dex  std={std_tng:.4f}  dex")
print(f"Physical offset: {mu_flam - mu_tng:.4f} dex  "
      f"(={10**(mu_flam-mu_tng):.3f}x enrichment difference)")

# The v3 feature cache is saved by reconstruction; verify the actual features.
with open(FEATURE_CACHE_DIR / "metallicity_log_zscore.pkl", "rb") as f:
    frozen_met = pickle.load(f)
np.testing.assert_allclose(frozen_met["flamingo_met_dex"], flam_met_dex, equal_nan=True)
np.testing.assert_allclose(frozen_met["tng_met_dex"], tng_met_dex, equal_nan=True)
print("Validated frozen v3 dex features; descriptive scalers above are for this plot.")

> Historical interpretation from before the corrected comparison rerun. Reassess quantitative and causal claims using the new outputs.

Metallicity is a bit risky to use given the higher standard deviation of TNG300 metallicity. The difference seems to be because of simulation differences.


### Velocity Dispersion Verification


> Historical interpretation from before the corrected comparison rerun. Reassess quantitative and causal claims using the new outputs.

First we test the inbuild fields for disperse velocity at pair level and compare histograms and CDFS.
stellar disperse velocity similar to FLAMINGO for TNG300 is created using the script compute_sigma_star.py where you will need to add your tng api key. Following is the TNG300 implementation and the one following it computed using the script. The different plots for both are shown.


In [ ]:
# FLAMINGO stellar vel disp  pair level
flam_vd_pairs = np.concatenate([
    sample["vel_disp"][pairs["i"]],
    sample["vel_disp"][pairs["j"]]
])
flam_vd_pairs = flam_vd_pairs[np.isfinite(flam_vd_pairs) & (flam_vd_pairs > 0)]

# TNG all-particle vel disp — pair level
tng_vd_pairs  = np.concatenate([
    tng_vd_sample[tng_pairs_i],
    tng_vd_sample[tng_pairs_j]
])
tng_vd_pairs = tng_vd_pairs[np.isfinite(tng_vd_pairs) & (tng_vd_pairs > 0)]

print(f"\nFLAMINGO vel_disp (stellar only):")
print(f"  N:      {len(flam_vd_pairs):,}")
print(f"  Median: {np.median(flam_vd_pairs):.1f} km/s")
print(f"  Range:  {flam_vd_pairs.min():.1f} – {flam_vd_pairs.max():.1f} km/s")

print(f"\nTNG SubhaloVelDisp (all particles):")
print(f"  N:      {len(tng_vd_pairs):,}")
print(f"  Median: {np.median(tng_vd_pairs):.1f} km/s")
print(f"  Range:  {tng_vd_pairs.min():.1f} – {tng_vd_pairs.max():.1f} km/s")

print(f"\nDefinition note:")
print(f"  FLAMINGO: sqrt((σ_xx + σ_yy + σ_zz)/3) — stellar particles only")
print(f"  TNG:      SubhaloVelDisp — all bound particles (DM+gas+stars)")

# plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Velocity Dispersion Verification — Pair-member entries", fontsize=12)

bins_vd = np.linspace(0, 400, 40)
axes[0].hist(flam_vd_pairs, bins=bins_vd, density=True,
             alpha=0.6, color="#E07B3F",
             label=f"FLAMINGO stellar (N={len(flam_vd_pairs):,})")
axes[0].hist(tng_vd_pairs,  bins=bins_vd, density=True,
             alpha=0.6, color="#4C72B0",
             label=f"TNG all-particle (N={len(tng_vd_pairs):,})")
axes[0].set_xlabel("Velocity Dispersion [km/s]")
axes[0].set_ylabel("Density")
axes[0].set_title("Distribution\n(different definitions)")
axes[0].legend(fontsize=8)

flam_vd_s = np.sort(flam_vd_pairs)
tng_vd_s  = np.sort(tng_vd_pairs)
axes[1].step(flam_vd_s, np.arange(1, len(flam_vd_s)+1)/len(flam_vd_s),
             color="#E07B3F", label="FLAMINGO stellar", lw=1.5, where="post")
axes[1].step(tng_vd_s,  np.arange(1, len(tng_vd_s)+1)/len(tng_vd_s),
             color="#4C72B0", label="TNG all-particle", lw=1.5, where="post")
axes[1].set_xlabel("Velocity Dispersion [km/s]")
axes[1].set_ylabel("CDF")
axes[1].set_title("CDF")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_DIR / "velocity_dispersion_catalogue_definition.pdf", dpi=150, bbox_inches="tight")
plt.show()

# KS + Wasserstein
ks, _ = ks_2samp(flam_vd_pairs, tng_vd_pairs)
wd    = wasserstein_distance(flam_vd_pairs, tng_vd_pairs)
pooled_std = np.sqrt(
    (len(flam_vd_pairs) * np.std(flam_vd_pairs)**2 +
     len(tng_vd_pairs)  * np.std(tng_vd_pairs)**2) /
    (len(flam_vd_pairs) + len(tng_vd_pairs))
)
print(f"\nKS stat:              {ks:.4f}")
print(f"Wasserstein (raw):    {wd:.2f} km/s")
print(f"Wasserstein (norm):   {wd/pooled_std:.4f}")

Same definition mass-weighted CoM velocity no aperture cut verified from FLAMINGO'S metadata and modified TNG300 accordingly. sqrt((σ_xx + σ_yy + σ_zz)/3)


In [ ]:
# FLAMINGO stellar vel disp — pair level
flam_vd_pairs = np.concatenate([
    sample["vel_disp"][pairs["i"]],
    sample["vel_disp"][pairs["j"]]
])
flam_vd_pairs = flam_vd_pairs[np.isfinite(flam_vd_pairs) & (flam_vd_pairs > 0)]

# TNG definition-matched stellar vel disp (mass-weighted, no aperture cut — FLAMINGO-like)  
tng_pairs_i_realid = keep_idx[tng_pairs_i]
tng_pairs_j_realid = keep_idx[tng_pairs_j]

tng_vd_pairs = np.concatenate([
    finite_lookup(tng_sigma_star, tng_pairs_i_realid, "tng_sigma_star"),
    finite_lookup(tng_sigma_star, tng_pairs_j_realid, "tng_sigma_star"),
])
tng_vd_pairs = tng_vd_pairs[np.isfinite(tng_vd_pairs) & (tng_vd_pairs > 0)]

print(f"\nFLAMINGO vel_disp (stellar only):")
print(f"  N:      {len(flam_vd_pairs):,}")
print(f"  Median: {np.median(flam_vd_pairs):.1f} km/s")
print(f"  Range:  {flam_vd_pairs.min():.1f} – {flam_vd_pairs.max():.1f} km/s")

print(f"\nTNG stellar vel_disp:")
print(f"  N:      {len(tng_vd_pairs):,}")
print(f"  Median: {np.median(tng_vd_pairs):.1f} km/s")
print(f"  Range:  {tng_vd_pairs.min():.1f} – {tng_vd_pairs.max():.1f} km/s")

# plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Velocity Dispersion Verification — Pair-member entries", fontsize=12)

bins_vd = np.linspace(0, 400, 40)
axes[0].hist(flam_vd_pairs, bins=bins_vd, density=True,
             alpha=0.6, color="#E07B3F",
             label=f"FLAMINGO stellar (N={len(flam_vd_pairs):,})")
axes[0].hist(tng_vd_pairs,  bins=bins_vd, density=True,
             alpha=0.6, color="#4C72B0",
             label=f"TNG all-bound stars (N={len(tng_vd_pairs):,})")
axes[0].set_xlabel("Velocity Dispersion [km/s]")
axes[0].set_ylabel("Density")
axes[0].set_title("Distribution")
axes[0].legend(fontsize=8)

flam_vd_s = np.sort(flam_vd_pairs)
tng_vd_s  = np.sort(tng_vd_pairs)
axes[1].step(flam_vd_s, np.arange(1, len(flam_vd_s)+1)/len(flam_vd_s),
             color="#E07B3F", label="FLAMINGO stellar", lw=1.5, where="post")
axes[1].step(tng_vd_s,  np.arange(1, len(tng_vd_s)+1)/len(tng_vd_s),
             color="#4C72B0", label="TNG all-bound stars", lw=1.5, where="post")
axes[1].set_xlabel("Velocity Dispersion [km/s]")
axes[1].set_ylabel("CDF")
axes[1].set_title("CDF")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_DIR / "velocity_dispersion_matched_stars.pdf", bbox_inches="tight")
plt.show()

# KS + Wasserstein
ks, _ = ks_2samp(flam_vd_pairs, tng_vd_pairs)
wd    = wasserstein_distance(flam_vd_pairs, tng_vd_pairs)
pooled_std = np.sqrt(
    (len(flam_vd_pairs) * np.std(flam_vd_pairs)**2 +
     len(tng_vd_pairs)  * np.std(tng_vd_pairs)**2) /
    (len(flam_vd_pairs) + len(tng_vd_pairs))
)
print(f"\nKS stat:              {ks:.4f}")
print(f"Wasserstein (raw):    {wd:.2f} km/s")
print(f"Wasserstein (norm):   {wd/pooled_std:.4f}")

### Morphology


> Historical interpretation from before the corrected comparison rerun. Reassess quantitative and causal claims using the new outputs.

First we verify the metadata of kappa_rot from FLAMINGO to make a TNG300 equivalent since a similar field does not exist in TNG300.


In [ ]:
if RUN_SOAP_METADATA:
    print("FLAMINGO BoundSubhalo/KappaCorotStars attributes:")
    for attr in soap[FIELD_PATHS["kappa_rot"]].attrs:
        print(f"  {attr}: {soap[FIELD_PATHS['kappa_rot']].attrs[attr]}")
else:
    print('SOAP kappa metadata skipped (optional).')

kappacorotstars is dimensionless no a factor or h factor. Positions are relative to centre of potential and velocities are relative to stellar centre of mass velocity. compute_kappa_corot.py does all the above and store the results in a pickle file.


In [ ]:
# loading the pkl file
with open(FEATURE_CACHE_DIR / "tng_kappa_corot.pkl", "rb") as f:
    kappa_cache = pickle.load(f)
print(f"Loaded: {len(kappa_cache):,} subhalos")

In [ ]:
# kappa_corot: FLAMINGO vs TNG comparison
from scipy import stats

#pair-level arrays
# FLAMINGO flam_kap_pairs

# TNG: map kappa_cache (keyed by global subhalo ID) to pair level
tng_kap_i = finite_lookup(kappa_cache, keep_idx[tng_pairs_i], "kappa_cache")
tng_kap_j = finite_lookup(kappa_cache, keep_idx[tng_pairs_j], "kappa_cache")
tng_kap_pairs = np.concatenate([tng_kap_i, tng_kap_j])
tng_kap_pairs = tng_kap_pairs[np.isfinite(tng_kap_pairs) & (tng_kap_pairs >= 0) & (tng_kap_pairs <= 1)]

# statistics
ks_stat, ks_p     = stats.ks_2samp(flam_kap_pairs, tng_kap_pairs)
wass_raw          = stats.wasserstein_distance(flam_kap_pairs, tng_kap_pairs)
combined_std      = np.sqrt(0.5 * (np.std(flam_kap_pairs)**2 + np.std(tng_kap_pairs)**2))
wass_norm         = normalized_wasserstein(flam_kap_pairs, tng_kap_pairs)

flam_disc_frac    = np.mean(flam_kap_pairs > 0.35)
tng_disc_frac     = np.mean(tng_kap_pairs  > 0.35)

print(f"FLAMINGO κ_co — N: {len(flam_kap_pairs):,}  Median: {np.median(flam_kap_pairs):.4f}  Disc frac: {flam_disc_frac:.3f}")
print(f"TNG      κ_co — N: {len(tng_kap_pairs):,}   Median: {np.median(tng_kap_pairs):.4f}   Disc frac: {tng_disc_frac:.3f}")
print(f"\nKS stat:              {ks_stat:.4f}  (descriptive distance; shared members)")
print(f"Wasserstein (raw):    {wass_raw:.4f}")
print(f"Wasserstein (norm):   {wass_norm:.4f}")

# plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("κ_co Verification — Pair-member entries", fontsize=13)

bins = np.linspace(0, 1, 40)

# histogram
ax = axes[0]
ax.set_title("Histogram")
ax.hist(flam_kap_pairs, bins=bins, density=True, alpha=0.5, color="tab:orange",
        label=f"FLAMINGO (N={len(flam_kap_pairs):,})")
ax.hist(tng_kap_pairs,  bins=bins, density=True, alpha=0.5, color="tab:blue",
        label=f"TNG300 (N={len(tng_kap_pairs):,})")
ax.axvline(0.35, color="k", ls="--", lw=1, label="disc threshold (0.35)")
ax.set_xlabel("κ_co")
ax.set_ylabel("Density")
ax.legend(fontsize=8)

# normalised histogram
ax = axes[1]
ax.set_title("Normalised")
ax.hist(flam_kap_pairs, bins=bins, density=True, alpha=0.5, color="tab:orange", label="FLAMINGO")
ax.hist(tng_kap_pairs,  bins=bins, density=True, alpha=0.5, color="tab:blue",  label="TNG300")
ax.axvline(np.median(flam_kap_pairs), color="tab:orange", ls=":", lw=1.5)
ax.axvline(np.median(tng_kap_pairs),  color="tab:blue",   ls=":", lw=1.5)
ax.axvline(0.35, color="k", ls="--", lw=1)
ax.set_xlabel("κ_co")
ax.legend(fontsize=8)

# CDF
ax = axes[2]
ax.set_title("CDF")
for arr, col, lbl in [(flam_kap_pairs, "tab:orange", "FLAMINGO"),
                       (tng_kap_pairs,  "tab:blue",   "TNG300")]:
    xs = np.sort(arr)
    ys = np.arange(1, len(xs) + 1) / len(xs)
    ax.step(xs, ys, where="post", color=col, label=lbl)
ax.axvline(0.35, color="k", ls="--", lw=1, label="disc threshold")
ax.set_xlabel("κ_co")
ax.set_ylabel("CDF")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_34.pdf", bbox_inches="tight")
plt.show()

> Historical interpretation from before the corrected comparison rerun. Reassess quantitative and causal claims using the new outputs.

Checked the exact way it is computed from github as well and my script seems to match the FLAMINGO implementation. I believe the difference is now due to FLAMINGO's kinetic SN feedback model and TNG's wind feedback.


In [ ]:
#  Baldry-style κ_co 
from scipy.stats import gaussian_kde
import matplotlib.colors as mcolors

# build (mass, kappa) pairs
flam_kap_bal  = np.concatenate([sample["kappa_rot"][pairs["i"]],    sample["kappa_rot"][pairs["j"]]])
flam_mass_kap = np.concatenate([sample["stellar_mass"][pairs["i"]], sample["stellar_mass"][pairs["j"]]])
valid_f = np.isfinite(flam_kap_bal) & np.isfinite(flam_mass_kap) & (flam_mass_kap > 0)
flam_kap_bal   = flam_kap_bal[valid_f]
flam_logm_kap  = np.log10(flam_mass_kap[valid_f])

tng_kap_i_full = finite_lookup(kappa_cache, keep_idx[tng_pairs_i], "kappa_cache")
tng_kap_j_full = finite_lookup(kappa_cache, keep_idx[tng_pairs_j], "kappa_cache")
tng_kap_bal    = np.concatenate([tng_kap_i_full,          tng_kap_j_full])
tng_mass_kap   = np.concatenate([tng_mass_sample[tng_pairs_i], tng_mass_sample[tng_pairs_j]])
valid_t = np.isfinite(tng_kap_bal) & np.isfinite(tng_mass_kap) & (tng_mass_kap > 0) & (tng_kap_bal >= 0) & (tng_kap_bal <= 1)
tng_kap_bal    = tng_kap_bal[valid_t]
tng_logm_kap   = np.log10(tng_mass_kap[valid_t])

x_grid = np.linspace(10.2, 11.8, 100)
y_grid = np.linspace(0.0,  1.0,  100)
X, Y   = np.meshgrid(x_grid, y_grid)
rng    = np.random.default_rng(42)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(r"Baldry-style $\kappa_{\rm co}$ on the y-axis and stellar mass on the x-axis", fontsize=13)

datasets = [
    (axes[0], flam_logm_kap, flam_kap_bal, "#E07B3F", "FLAMINGO L1_m8", 50_000, 0.12),
    (axes[1], tng_logm_kap,  tng_kap_bal,  "#4C72B0", "TNG300-1",       len(tng_logm_kap), 0.20),
]

for ax, log_mass, kap, color, label, n_kde, bw in datasets:
    hb = ax.hexbin(log_mass, kap, gridsize=50,
                   cmap="Greys", mincnt=1,
                   norm=mcolors.LogNorm(),
                   linewidths=0.2)

    idx = rng.choice(len(log_mass), min(n_kde, len(log_mass)), replace=False)
    kde = gaussian_kde(
        np.vstack([log_mass[idx], kap[idx]]),
        bw_method=bw
    )
    Z      = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)
    levels = np.percentile(Z[Z > 0], [50, 70, 88, 95])
    ax.contour(X, Y, Z, levels=levels,
               colors=color, linewidths=1.2, alpha=0.85)

    ax.axhline(0.28, color="gray", linestyle="--",
               linewidth=1.0, alpha=0.7, label=r"$\kappa_{\rm co} = 0.28$")

    # MW and M31 approximate κ_co (disc galaxies; values from Correa+2017 range)
    ax.scatter(np.log10(5.5e10), 0.52, marker="*", s=200,
               color="gold", edgecolors="black", linewidths=0.5,
               zorder=5, label="Illustrative MW-like value (0.52)")
    ax.scatter(np.log10(1.0e11), 0.40, marker="*", s=200,
               color="silver", edgecolors="black", linewidths=0.5,
               zorder=5, label="Illustrative M31-like value (0.40)")

    ax.set_xlabel(r"$\log_{10}(M_*/M_\odot)$", fontsize=11)
    ax.set_ylabel(r"$\kappa_{\rm co}$", fontsize=11)
    ax.set_title(f"{label}  (N={len(kap):,})", fontsize=11)
    ax.set_xlim(10.2, 11.8)
    ax.set_ylim(0.0,  1.0)
    ax.legend(fontsize=9, loc="upper left")
    plt.colorbar(hb, ax=ax, label="Counts per hexbin (log scale)")

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_36.pdf", bbox_inches="tight")
plt.show()

> Historical interpretation from before the corrected comparison rerun. Reassess quantitative and causal claims using the new outputs.

FLAMINGO's kinetic SN + AGN feedback model suppresses disc formation far more effectively than TNG's wind model in the MW-mass regime, producing a near-unimodal spheroidal k_co distribution vs TNG's bimodal one. FLAMINGO has a disc fraction of 0.285 and TNG300 has a disc fraction of 0.478.This gives a quantitative figure to the above conclusion.


Used a value of 0.28 to seperate the two populations in TNG300.


In [ ]:
from flamingo_diagnostics import print_full_report

if RUN_PARENT_CATALOGUE_CHECKS:
    with open(cache_dir / "flamingo_fields.pkl", "rb") as f:
        fields = pickle.load(f)

    flamingo_diag = print_full_report(fields, sample)
else:
    print(
        "Full FLAMINGO parent-catalogue diagnostics skipped. "
        "Set RUN_PARENT_CATALOGUE_CHECKS = True in the setup cell to run them."
    )